# Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
import re
import importlib
import CFDUtils
importlib.reload(CFDUtils)
from CFDUtils import dataExtraction

# Data frame configurations

In [ ]:
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
pd.options.display.float_format = '{:.6g}'.format

# Project directories

In [ ]:
currentDir = Path.cwd().resolve()

projectDir = next(
    (path for path in [currentDir, *currentDir.parents] if (path / '.git').exists()),
    None
)

openfoamProjectDir = Path(r'/path/to/OpenFOAM/run/airfoilCFDML')

# Experimental data

In [ ]:
expDataPath = projectDir/'CFD'/'data'/'expData.dat'
    
df = pd.read_csv(expDataPath, skiprows= 7, sep = r'\s+', header = None)
expData = df[:6].to_numpy(dtype=float)

# Mesh convergence

In [ ]:
# dictionaries for storing information
CdData = {
    'cells' : [17852, 35982, 68963, 140128], 
    'Cd' : [], 
    'CdCoV' : [],
    'CdChange%' : [], 
    'time' : [],
    'timeChange%': []
}

ClData = {
    'cells' : [17852, 35982, 68963, 140128], 
    'Cl' : [], 
    'ClCoV' : [],
    'ClChange%' : [],
    'time' : [],
    'timeChange%' : []
}

# extract CFD data from OpenFOAM
for meshLevel in range (1, 5):
    # forces and coefficients of variance
    forcePath = openfoamProjectDir/'meshConvergence'/f'mesh{meshLevel}'/'postProcessing'/'forceCoeffs'/'0'/'coefficient.dat'

    df = pd.read_csv(forcePath, skiprows= 13, sep = r'\s+', header = None).to_numpy()
    forceData = np.column_stack((df[:, 0], df[:, 1], df[:, 4]))

    CdData['Cd'].append(np.average(forceData[-200:, 1]))
    CdData['CdCoV'].append(np.std(forceData[-200:, 1])/np.average(forceData[-200:, 1]))
    ClData['Cl'].append(np.average(forceData[-200:, 2]))
    ClData['ClCoV'].append(np.std(forceData[-200:, 2])/np.average(forceData[-200:, 2]))

    # run time
    simpleFoamLog = openfoamProjectDir/'meshConvergence'/f'mesh{meshLevel}'/'log.simpleFoam'

    matches = re.findall(r"ExecutionTime\s*=\s*[0-9.eE+-]+\s*s\s+ClockTime\s*=\s*([0-9.eE+-]+)\s*s", simpleFoamLog.read_text(encoding="utf-8",errors="ignore"))
    CdData['time'].append (float(matches[-1]))
    ClData['time'].append (float(matches[-1]))

# calculate percentage changes
CdData['CdChange%'] = ['/'] + list((np.array(CdData['Cd'][1:]) - np.array(CdData['Cd'][:-1]))/np.array(CdData['Cd'][:-1])*100)
CdData['timeChange%'] = ['/'] + list((np.array(CdData['time'][1:]) - np.array(CdData['time'][:-1]))/np.array(CdData['time'][:-1])*100)

ClData['ClChange%'] = ['/'] + list((np.array(ClData['Cl'][1:]) - np.array(ClData['Cl'][:-1]))/np.array(ClData['Cl'][:-1])*100)
ClData['timeChange%'] = ['/'] + list((np.array(ClData['time'][1:]) - np.array(ClData['time'][:-1]))/np.array(ClData['time'][:-1])*100)

# convert to dataframes for printing
CdData = pd.DataFrame(CdData)
ClData = pd.DataFrame(ClData)

# presentation
display(HTML("<h1>Cd Mesh Convergence</h1>"))

print (CdData.to_string(index = False, col_space = 20))

figCdConverge = plt.figure(figsize=(15, 5))
ax1 = figCdConverge.add_subplot(1, 1, 1)
ax1.minorticks_on()
ax1.xaxis.set_minor_locator(plt.NullLocator())
ax1.set_xticks(CdData['cells'])
ax1.grid(True, which='major', linestyle='-', linewidth=0.8)
ax1.grid(True, which='minor', linestyle='--', linewidth=0.4)
ax1.plot(CdData['cells'], CdData['Cd'])
ax1.set_xlabel('Number of Cells')
ax1.set_ylabel('$C_D$')
plt.show()

display(HTML("<h1>Cl Mesh Convergence</h1>"))

print (ClData.to_string(index = False, col_space = 20))

figClConverge = plt.figure(figsize=(15, 5))
ax1 = figClConverge.add_subplot(1, 1, 1)
ax1.minorticks_on()
ax1.xaxis.set_minor_locator(plt.NullLocator())
ax1.set_xticks(ClData['cells'])
ax1.grid(True, which='major', linestyle='-', linewidth=0.8)
ax1.grid(True, which='minor', linestyle='--', linewidth=0.4)
ax1.plot(ClData['cells'], ClData['Cl'])
ax1.set_xlabel('Number of Cells')
ax1.set_ylabel('$C_L$')
plt.show()

# Validation

In [ ]:
# preparing dictionaries for information storage
AoADeg = expData[:, 0]

CdMesh3Data = {
    'AoA' : AoADeg,
    'CFD' : [],
    'experimental' : expData[:, 2], 
    'absoluteError' : [],
    'relativeError%' : [],
    'time' : []
}

ClMesh3Data = {
    'AoA' : AoADeg,
    'CFD' : [],
    'experimental' : expData[:, 1], 
    'absoluteError' : [],
    'relativeError%' : [],
    'time' : []
}

# extract CFD data at each AoA
for A in range(len(AoADeg)):
    # forces
    if AoADeg[A] < 0:
        forcePath = openfoamProjectDir/'validationMesh3'/f'n{-AoADeg[A]:.2f}'/'postProcessing'/'forceCoeffs'/'0'/'coefficient.dat'
    else:
        forcePath = openfoamProjectDir/'validationMesh3'/f'{AoADeg[A]:.2f}'/'postProcessing'/'forceCoeffs'/'0'/'coefficient.dat'        
    
    df = pd.read_csv(forcePath, skiprows= 13, sep = r'\s+', header = None).to_numpy()
    forceData = np.column_stack((df[:, 0], df[:, 1], df[:, 4]))

    CdMesh3Data['CFD'].append(np.average(forceData[-200:, 1]))
    ClMesh3Data['CFD'].append(np.average(forceData[-200:, 2]))

    # time 
    if AoADeg[A] < 0:
        simpleFoamLog = openfoamProjectDir/'validationMesh3'/f'n{-AoADeg[A]:.2f}'/'log.simpleFoam'
    else:
        simpleFoamLog = openfoamProjectDir/'validationMesh3'/f'{AoADeg[A]:.2f}'/'log.simpleFoam'

    matches = re.findall(r"ExecutionTime\s*=\s*[0-9.eE+-]+\s*s\s+ClockTime\s*=\s*([0-9.eE+-]+)\s*s", simpleFoamLog.read_text(encoding="utf-8",errors="ignore"))
    CdMesh3Data['time'].append(float(matches[-1]))
    ClMesh3Data['time'].append(float(matches[-1]))

# error calculation
CdMesh3Data['absoluteError'] = list(np.abs(np.array(CdMesh3Data['CFD']) - np.array(CdMesh3Data['experimental'])))
CdMesh3Data['relativeError%'] = list(np.abs(np.array(CdMesh3Data['absoluteError'])/np.array(CdMesh3Data['experimental'])*100))

ClMesh3Data['absoluteError'] = list(np.abs(np.array(ClMesh3Data['CFD']) - np.array(ClMesh3Data['experimental'])))
ClMesh3Data['relativeError%'] = list(np.abs(np.array(ClMesh3Data['absoluteError'])/np.array(ClMesh3Data['experimental'])*100))

# convert to data frames
CdMesh3Data = pd.DataFrame(CdMesh3Data)
ClMesh3Data = pd.DataFrame(ClMesh3Data)

# presentation
display(HTML("<h1>Mesh3</h1>"))

mesh3RunTime = sum(CdMesh3Data['time'])
mesh3CdAvgError = np.average(CdMesh3Data['relativeError%'])
mesh3ClAvgError = np.average(ClMesh3Data['relativeError%'][1:])

display(HTML("<h2>Cd Vs AoA</h2>"))

print (CdMesh3Data.to_string(index = False, col_space = 20))

figCd = plt.figure(figsize=(12, 8))
ax1 = figCd.add_subplot(1, 1, 1)
ax1.minorticks_on()
ax1.grid(True, which='major', linestyle='-', linewidth=0.8)
ax1.grid(True, which='minor', linestyle='--', linewidth=0.4)
ax1.plot(AoADeg, CdMesh3Data['CFD'], color = 'black', label = 'CFD results')
ax1.scatter(AoADeg, expData[:, 2], marker = 'x', color = 'blue', label = 'Experimental data')
ax1.set_xlabel('$AoA$ (degrees)')
ax1.set_ylabel('$C_D$')
ax1.legend ()
plt.show()

display(HTML("<h2>Cl Vs AoA</h2>"))

print (ClMesh3Data.to_string(index = False, col_space = 20))

figCl = plt.figure(figsize=(12, 8))
ax1 = figCl.add_subplot(1, 1, 1)
ax1.minorticks_on()
ax1.grid(True, which='major', linestyle='-', linewidth=0.8)
ax1.grid(True, which='minor', linestyle='--', linewidth=0.4)
ax1.plot(AoADeg, ClMesh3Data['CFD'], color = 'black', label = 'CFD results')
ax1.scatter(AoADeg, expData[:, 1], marker = 'x', color = 'blue', label = 'Experimental data')
ax1.set_xlabel('$AoA$ (degrees)')
ax1.set_ylabel('$C_L$')
ax1.legend ()
plt.show()

In [ ]:
# preparing dictionaries for storing information
AoADeg = expData[:, 0]

CdMesh4Data = {
    'AoA' : AoADeg,
    'CFD' : [],
    'experimental' : expData[:, 2], 
    'absoluteError' : [],
    'relativeError%' : [],
    'time' : []
}

ClMesh4Data = {
    'AoA' : AoADeg,
    'CFD' : [],
    'experimental' : expData[:, 1], 
    'absoluteError' : [],
    'relativeError%' : [],
    'time' : []
}

# extract CFD data at each AoA
for A in range(len(AoADeg)):
    # forces
    if AoADeg[A] < 0:
        forcePath = openfoamProjectDir/'validationMesh4'/f'n{-AoADeg[A]:.2f}'/'postProcessing'/'forceCoeffs'/'0'/'coefficient.dat'
    else:
        forcePath = openfoamProjectDir/'validationMesh4'/f'{AoADeg[A]:.2f}'/'postProcessing'/'forceCoeffs'/'0'/'coefficient.dat'        
    
    df = pd.read_csv(forcePath, skiprows= 13, sep = r'\s+', header = None).to_numpy()
    forceData = np.column_stack((df[:, 0], df[:, 1], df[:, 4]))

    CdMesh4Data['CFD'].append(np.average(forceData[-200:, 1]))
    ClMesh4Data['CFD'].append(np.average(forceData[-200:, 2]))

    # time
    if AoADeg[A] < 0:
        simpleFoamLog = openfoamProjectDir/'validationMesh4'/f'n{-AoADeg[A]:.2f}'/'log.simpleFoam'
    else:
        simpleFoamLog = openfoamProjectDir/'validationMesh4'/f'{AoADeg[A]:.2f}'/'log.simpleFoam'

    matches = re.findall(r"ExecutionTime\s*=\s*[0-9.eE+-]+\s*s\s+ClockTime\s*=\s*([0-9.eE+-]+)\s*s", simpleFoamLog.read_text(encoding="utf-8",errors="ignore"))
    CdMesh4Data['time'].append(float(matches[-1]))
    ClMesh4Data['time'].append(float(matches[-1]))

# calculate errors
CdMesh4Data['absoluteError'] = list(np.abs(np.array(CdMesh4Data['CFD']) - np.array(CdMesh4Data['experimental'])))
CdMesh4Data['relativeError%'] = list(np.abs(np.array(CdMesh4Data['absoluteError'])/np.array(CdMesh4Data['experimental'])*100))

ClMesh4Data['absoluteError'] = list(np.abs(np.array(ClMesh4Data['CFD']) - np.array(ClMesh4Data['experimental'])))
ClMesh4Data['relativeError%'] = list(np.abs(np.array(ClMesh4Data['absoluteError'])/np.array(ClMesh4Data['experimental'])*100))

# convert to data frames
CdMesh4Data = pd.DataFrame(CdMesh4Data)
ClMesh4Data = pd.DataFrame(ClMesh4Data)

# presentation
display(HTML("<h1>Mesh4</h1>"))

mesh4RunTime = sum(CdMesh4Data['time'])
mesh4CdAvgError = np.average(CdMesh4Data['relativeError%'])
mesh4ClAvgError = np.average(ClMesh4Data['relativeError%'][1:])

display(HTML("<h2>Cd Vs AoA</h2>"))

print (CdMesh4Data.to_string(index = False, col_space = 20))

figCd = plt.figure(figsize=(12, 8))
ax1 = figCd.add_subplot(1, 1, 1)
ax1.minorticks_on()
ax1.grid(True, which='major', linestyle='-', linewidth=0.8)
ax1.grid(True, which='minor', linestyle='--', linewidth=0.4)
ax1.plot(AoADeg, CdMesh4Data['CFD'], color = 'black', label = 'CFD results')
ax1.scatter(AoADeg, expData[:, 2], marker = 'x', color = 'blue', label = 'Experimental data')
ax1.set_xlabel('$AoA$ (degrees)')
ax1.set_ylabel('$C_D$')
ax1.legend ()
plt.show()

display(HTML("<h2>Cl Vs AoA</h2>"))

print (ClMesh4Data.to_string(index = False, col_space = 20))

figCl = plt.figure(figsize=(12, 8))
ax1 = figCl.add_subplot(1, 1, 1)
ax1.minorticks_on()
ax1.grid(True, which='major', linestyle='-', linewidth=0.8)
ax1.grid(True, which='minor', linestyle='--', linewidth=0.4)
ax1.plot(AoADeg, ClMesh4Data['CFD'], color = 'black', label = 'CFD results')
ax1.scatter(AoADeg, expData[:, 1], marker = 'x', color = 'blue', label = 'Experimental data')
ax1.set_xlabel('$AoA$ (degrees)')
ax1.set_ylabel('$C_L$')
ax1.legend ()
plt.show()

# Mesh 3 and 4 comparison

In [ ]:
meshComparison = pd.DataFrame({
    'mesh' : [3, 4],
    'avgCdError%' : [mesh3CdAvgError, mesh4CdAvgError],
    'Cd%PointChange' : ['/', mesh4CdAvgError-mesh3CdAvgError],
    'avgClError%' : [mesh3ClAvgError, mesh4ClAvgError],
    'Cl%PointChange' : ['/', mesh4ClAvgError-mesh3ClAvgError],
    'runTime' : [mesh3RunTime, mesh4RunTime],
    'runTimeChange%' : ['/', (mesh4RunTime-mesh3RunTime)/mesh3RunTime*100]
})

print (meshComparison.to_string(index = False, col_space = 15))

# CFD training data extraction

In [ ]:
trainingDataPath = projectDir/'ML'/'data'/'trainingData.txt'
trainingDataPath.parent.mkdir(parents=True, exist_ok=True)

trainingSetPath = projectDir/'ML'/'data'/'trainingSet.txt'

CFDPath = openfoamProjectDir/'ML'

dataExtraction (trainingDataPath, trainingSetPath, CFDPath)

# CFD training data information

In [ ]:
trainingDataPath = projectDir/'ML'/'data'/'trainingData.txt'
trainingData = pd.read_csv(trainingDataPath, sep = '\t')

print (trainingData.describe())

# CFD held-out test data extraction

In [ ]:
testDataPath = projectDir/'ML'/'data'/'testData.txt'
testDataPath.parent.mkdir(parents=True, exist_ok=True)

testSetPath = projectDir/'ML'/'data'/'testSet.txt'

CFDPath = openfoamProjectDir/'MLTest'

dataExtraction (testDataPath, testSetPath, CFDPath)